<a href="https://colab.research.google.com/github/FridaOyucho/HTS-Model-/blob/main/xgboostmodel_functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HTS XGBoost Model – Function-Based Pipeline
Every section of the original notebook (`xGBoostModelreviewed`) is now a reusable function.
Run the cells top to bottom to define the functions, then run the last cell (`run_pipeline()`).

In [ ]:
# Uncomment in Colab if needed
# !pip install --upgrade pandas numpy scikit-learn xgboost statsmodels openpyxl

## 0. Imports and configuration

In [ ]:
import os
import time
from itertools import product

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy import sparse as scipy_sparse
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_curve,
)
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from statsmodels.stats.proportion import proportion_confint
from xgboost import XGBClassifier

# ---------------- Configuration ----------------
DATA_PATHS = {
    "sparse_train": "sparse_train (1).csv",
    "sparse_val": "sparse_val (1).csv",
    "sparse_test": "sparse_test (1).csv",
}

COLS_TO_DROP = [
    "EMR", "SDP", "PatientPk", "MFL_Code", "SiteCode", "Latitude", "Longitude",
    "Facility_Name", "SubCounty", "ClientTestedAs", "CoupleDiscordant",
    "VisitDate", "NumberOfPartners",
]

TARGET_COL = "FinalTestResult"
SENSITIVITY_TARGETS = (0.90, 0.95)            # used for validation operating points
TARGETS_PCT = {90: 0.90, 95: 0.95}            # used for county evaluation
MIN_POSITIVES = 50                            # counties above this get their own threshold
RANDOM_STATE = 2231

MODEL_DIR = "/content" if os.path.isdir("/content") else "."

HYPERPARAMETER_GRID = {
    "max_depth": [5],
    "learning_rate": [0.05],
    "scale_pos_weight": [1],
    "subsample": [0.8],
}

## 1. Data loading and cleaning

In [ ]:
def load_datasets(paths):
    """Read each CSV in `paths` (name -> file path) into a dict of DataFrames."""
    datasets = {name: pd.read_csv(path) for name, path in paths.items()}
    print(datasets.keys())
    for name, df in datasets.items():
        print(name, df.shape)
    return datasets


def drop_columns(datasets, cols_to_drop):
    """Drop identifier / leakage columns from every dataset (missing columns are ignored)."""
    return {
        name: df.drop(columns=cols_to_drop, errors="ignore")
        for name, df in datasets.items()
    }


def report_dtypes(datasets):
    """Print a count of data types for each dataset."""
    for name, df in datasets.items():
        print(f"\n{name}")
        print(df.dtypes.value_counts())


def cast_string_columns_to_object(datasets):
    """Convert pandas string-typed columns to plain object dtype (in place)."""
    for df in datasets.values():
        str_cols = [
            c for c in df.columns
            if pd.api.types.is_string_dtype(df[c]) and df[c].dtype != object
        ]
        if str_cols:
            df[str_cols] = df[str_cols].astype(object)
    return datasets

## 2. Train / validation / test sets

In [ ]:
def split_features_target(datasets, target_col=TARGET_COL):
    """Return X_train, y_train, X_val, y_val, X_test, y_test."""
    out = []
    for key in ("sparse_train", "sparse_val", "sparse_test"):
        df = datasets[key]
        out.extend([df.drop(columns=[target_col]), df[target_col]])
    return tuple(out)


def summarize_dtypes(X):
    """Table of variable names and data types, sorted by type."""
    return (
        pd.DataFrame({"Variable": X.columns, "Datatype": X.dtypes.astype(str)})
        .sort_values("Datatype")
    )


def print_unique_counts(X):
    """Print the number of unique values per column."""
    for col in X.columns:
        print(col, X[col].nunique())

## 3. Encode categorical features (sparse one-hot)

In [ ]:
def get_categorical_columns(X):
    """Names of non-numeric (categorical / text) columns."""
    return [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]


def build_encoder(X_train):
    """Create the one-hot ColumnTransformer for the categorical columns."""
    categorical_cols = get_categorical_columns(X_train)
    return ColumnTransformer(
        transformers=[
            ("categorical",
             OneHotEncoder(handle_unknown="ignore", dtype=np.float32),
             categorical_cols)
        ],
        remainder="passthrough",
        sparse_threshold=1.0,
    )


def encode_features(X_train, X_val, X_test):
    """Fit encoder on train, transform all splits into CSR sparse matrices."""
    encoder = build_encoder(X_train)
    X_train_enc = scipy_sparse.csr_matrix(encoder.fit_transform(X_train))
    X_val_enc = scipy_sparse.csr_matrix(encoder.transform(X_val))
    X_test_enc = scipy_sparse.csr_matrix(encoder.transform(X_test))
    feature_names = encoder.get_feature_names_out()

    print("Same features in train & val:", X_train_enc.shape[1] == X_val_enc.shape[1])
    print("Same features in train & test:", X_train_enc.shape[1] == X_test_enc.shape[1])
    return encoder, X_train_enc, X_val_enc, X_test_enc, feature_names


def preview_encoded(X_enc, feature_names, n_rows=5, n_cols=10):
    """Small dense preview of the sparse matrix."""
    return pd.DataFrame(
        X_enc[:n_rows, :n_cols].toarray(), columns=feature_names[:n_cols]
    )


def describe_sparse_matrix(X_enc):
    """Print shape, dtype, non-zero count and density; return density."""
    print("shape:", X_enc.shape)
    print("dtype:", X_enc.dtype)
    print("stored nonzeros:", X_enc.nnz)
    density = X_enc.nnz / (X_enc.shape[0] * X_enc.shape[1])
    print(f"Sparse matrix density: {density:.6f}")
    return density

## 4. Encode the target

In [ ]:
def encode_target(y_train, y_val, y_test):
    """Label-encode the target using the training-set mapping."""
    print(y_train.value_counts())
    label_encoder = LabelEncoder()
    y_train_enc = label_encoder.fit_transform(y_train)
    y_val_enc = label_encoder.transform(y_val)
    y_test_enc = label_encoder.transform(y_test)
    return label_encoder, y_train_enc, y_val_enc, y_test_enc


def compute_scale_pos_weight(y_train_enc):
    """negatives / positives – useful because HIV positivity is imbalanced."""
    neg = (y_train_enc == 0).sum()
    pos = (y_train_enc == 1).sum()
    value = neg / pos
    print(value)
    return value

## 5. Train XGBoost (grid search on validation AUPRC)

In [ ]:
def train_xgboost_grid(X_train_enc, y_train_enc, X_val_enc, y_val_enc,
                       grid=HYPERPARAMETER_GRID,
                       results_csv="xgb_hyperparameter_results.csv"):
    """
    Train one model per hyperparameter combination, keep the best by validation AUPRC.
    Returns (best_model, last_model, best_params, best_score, results_df).
    """
    configurations = list(product(
        grid["max_depth"], grid["learning_rate"],
        grid["scale_pos_weight"], grid["subsample"],
    ))
    total_runs = len(configurations)

    results, best_model, best_params = [], None, None
    best_score, last_model = float("-inf"), None
    search_start = time.perf_counter()

    for run_number, (max_depth, learning_rate, spw, subsample) in enumerate(configurations, start=1):
        print(
            f"Starting {run_number}/{total_runs} ({run_number / total_runs:.1%}): "
            f"depth={max_depth}, lr={learning_rate}, spw={spw}, subsample={subsample}",
            flush=True,
        )
        model = XGBClassifier(
            n_estimators=1000,
            max_depth=max_depth,
            learning_rate=learning_rate,
            scale_pos_weight=spw,
            subsample=subsample,
            random_state=RANDOM_STATE,
            eval_metric="aucpr",
            early_stopping_rounds=50,
            n_jobs=-1,
        )
        model.fit(X_train_enc, y_train_enc,
                  eval_set=[(X_val_enc, y_val_enc)], verbose=False)

        score = model.best_score
        params = {"max_depth": max_depth, "learning_rate": learning_rate,
                  "scale_pos_weight": spw, "subsample": subsample}
        results.append({**params, "validation_aucpr": score,
                        "best_iteration": model.best_iteration})
        last_model = model

        if score > best_score:
            best_score, best_model, best_params = score, model, params

        elapsed = time.perf_counter() - search_start
        eta = (elapsed / run_number) * (total_runs - run_number)
        print(f"Completed {run_number}/{total_runs}; AUPRC={score:.5f}; "
              f"elapsed={elapsed / 60:.1f} min; ETA={eta / 60:.1f} min", flush=True)

    results_df = (
        pd.DataFrame(results)
        .sort_values("validation_aucpr", ascending=False)
        .reset_index(drop=True)
    )
    results_df.to_csv(results_csv, index=False)

    print(f"Completed {len(results)} of {total_runs} configurations")
    print("Best parameters:", best_params)
    print("Best validation AUPRC:", best_score)
    print("Best iteration:", best_model.best_iteration)
    return best_model, last_model, best_params, best_score, results_df

## 6. Save, download and load models

In [ ]:
def download_file(path):
    """Trigger a browser download in Colab; otherwise just report the path."""
    try:
        from google.colab import files
        files.download(path)
    except ImportError:
        print(f"Not running in Colab – file saved at: {path}")


def save_models(models, directory=MODEL_DIR, download=True):
    """Save {filename: model} with joblib and optionally download them."""
    paths = []
    for filename, model in models.items():
        path = os.path.join(directory, filename)
        joblib.dump(model, path)
        paths.append(path)
        if download:
            download_file(path)
    return paths


def load_models(xgb_path, best_path):
    """Load previously saved models."""
    xgb_model = joblib.load(xgb_path)
    best_model = joblib.load(best_path)
    print(type(xgb_model))
    print(type(best_model))
    return xgb_model, best_model

## 7. Validation performance and thresholds (90% and 95% sensitivity)

In [ ]:
def evaluate_default_threshold(model, X_enc, y_enc):
    """Precision and recall at the model's default 0.5 cut-off."""
    y_pred = model.predict(X_enc)
    precision = precision_score(y_enc, y_pred)
    recall = recall_score(y_enc, y_pred)
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    return precision, recall


def find_operating_points(y_true, y_prob, targets=SENSITIVITY_TARGETS):
    """Highest threshold that still reaches each target sensitivity (precision-recall curve)."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    points = {}
    for target in targets:
        # Exclude final recall value because thresholds is one element shorter
        valid = np.where(recall[:-1] >= target)[0]
        idx = valid[-1]
        points[target] = {
            "threshold": thresholds[idx],
            "recall": recall[idx],
            "precision": precision[idx],
        }
        print(f"\nTarget sensitivity: {target:.0%}")
        print(f"Validation sensitivity: {recall[idx]:.4f}")
        print(f"Validation precision: {precision[idx]:.4f}")
        print(f"Selected threshold: {thresholds[idx]:.6f}")
    return points


def freeze_thresholds(operating_points):
    """Pull out the 90% and 95% thresholds."""
    threshold_90 = operating_points[0.90]["threshold"]
    threshold_95 = operating_points[0.95]["threshold"]
    print("90% threshold:", threshold_90)
    print("95% threshold:", threshold_95)
    return threshold_90, threshold_95


def build_global_thresholds_df(operating_points):
    """Summary table of the global (validation-derived) thresholds."""
    rows = [
        {
            "Target_Sensitivity_%": int(round(target * 100)),
            "Threshold": p["threshold"],
            "Validation_Sensitivity_%": p["recall"] * 100,
            "Validation_Precision_%": p["precision"] * 100,
        }
        for target, p in operating_points.items()
    ]
    return pd.DataFrame(rows).round(6)

## 8. Apply frozen thresholds to the test set

In [ ]:
def predict_at_threshold(y_prob, threshold):
    """Binary predictions at a given probability threshold."""
    return (y_prob >= threshold).astype(int)


def report_test_performance(y_test_enc, y_prob, threshold, label):
    """Print a classification report at the given threshold and return predictions."""
    y_pred = predict_at_threshold(y_prob, threshold)
    print(f"TEST PERFORMANCE - {label} TARGET")
    print(classification_report(y_test_enc, y_pred, digits=4))
    return y_pred


def test_positivity_and_yield(y_test_enc, y_prob, thresholds):
    """Positivity rate, plus sensitivity / precision / share of population tested per threshold."""
    positivity_rate = y_test_enc.mean()
    print(f"Test positivity rate: {positivity_rate:.4%}")

    for target, threshold in thresholds:
        y_pred = predict_at_threshold(y_prob, threshold)
        tn, fp, fn, tp = confusion_matrix(y_test_enc, y_pred).ravel()
        print(f"\nTarget sensitivity: {target:.0%}")
        print(f"Threshold: {threshold:.6f}")
        print(f"Actual test sensitivity: {tp / (tp + fn):.2%}")
        print(f"Precision: {tp / (tp + fp):.2%}")
        print(f"Population Tested: {(tp + fp) / len(y_test_enc):.2%}")
    return positivity_rate

## 9. Plots (PR curve and confusion matrices)

In [ ]:
def plot_pr_curve(y_test_enc, y_prob, threshold_90, threshold_95):
    """Precision-recall curve with the 90% and 95% operating points marked."""
    precision_test, recall_test, _ = precision_recall_curve(y_test_enc, y_prob)
    ap_test = average_precision_score(y_test_enc, y_prob)
    prevalence = np.mean(y_test_enc)

    y_pred_90 = predict_at_threshold(y_prob, threshold_90)
    y_pred_95 = predict_at_threshold(y_prob, threshold_95)
    recall_90 = recall_score(y_test_enc, y_pred_90)
    precision_90 = precision_score(y_test_enc, y_pred_90)
    recall_95 = recall_score(y_test_enc, y_pred_95)
    precision_95 = precision_score(y_test_enc, y_pred_95)

    plt.figure(figsize=(9, 7))
    plt.plot(recall_test, precision_test, linewidth=2,
             label=f"Test PR Curve (AP = {ap_test:.3f})")
    plt.axhline(y=prevalence, color="gray", linestyle="--", linewidth=1.5,
                label=f"Test positivity rate = {prevalence:.2%}")
    plt.scatter(recall_90, precision_90, color="red", s=120, zorder=5,
                label=f"90% target: Recall={recall_90:.2%}, Precision={precision_90:.2%}")
    plt.scatter(recall_95, precision_95, color="green", s=120, zorder=5,
                label=f"95% target: Recall={recall_95:.2%}, Precision={precision_95:.2%}")
    plt.axvline(recall_90, color="red", linestyle=":", alpha=0.6)
    plt.axvline(recall_95, color="green", linestyle=":", alpha=0.6)
    plt.xlabel("Recall (Sensitivity)")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve with 90% and 95% Sensitivity Operating Points")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    return y_pred_90, y_pred_95


def plot_confusion_matrix(y_true, y_pred, title, cmap="Blues"):
    """Confusion matrix plot; returns the matrix."""
    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(
        confusion_matrix=cm, display_labels=["Negative", "Positive"]
    ).plot(cmap=cmap, values_format=",d", colorbar=False)
    plt.title(title)
    plt.tight_layout()
    plt.show()
    return cm

## 10. Feature importance

In [ ]:
def summarize_feature_importance(encoder, model, exclude=("County",)):
    """Importance (%) per encoded feature and rolled up to the base variable."""
    importance_df = pd.DataFrame({
        "feature": encoder.get_feature_names_out(),
        "importance": model.feature_importances_,
    })
    importance_df["importance_pct"] = (
        importance_df["importance"] / importance_df["importance"].sum() * 100
    )
    importance_df["base_feature"] = (
        importance_df["feature"]
        .str.replace(r"^(categorical__|remainder__)", "", regex=True)
        .str.split("_")
        .str[0]
    )
    feature_summary = (
        importance_df.groupby("base_feature")["importance_pct"]
        .sum()
        .sort_values(ascending=False)
        .reset_index()
    )
    feature_summary = feature_summary[~feature_summary["base_feature"].isin(exclude)]
    return importance_df, feature_summary

## 11. Scored data frames (validation and test)

In [ ]:
def build_scored_frame(df, y_enc, model, X_enc):
    """Copy of `df` with `actual` (0/1) and `score` (predicted probability) columns."""
    scored = df.copy()
    scored["actual"] = np.asarray(y_enc)
    scored["score"] = model.predict_proba(X_enc)[:, 1]
    return scored


def check_scored_frame(scored, X_enc, y_enc):
    """Sanity checks that rows line up with the encoded matrices."""
    print("rows:", len(scored))
    print("X_enc rows:", X_enc.shape[0])
    print("y_enc rows:", len(y_enc))
    print("Positives (y):", np.asarray(y_enc).sum())
    print("Positives (frame):", scored["actual"].sum())
    print(scored[["County", "actual", "score"]].head())

## 12. Group-specific thresholds (Sex, Population type, County)

In [ ]:
def get_threshold(y, score, target):
    """First ROC point reaching `target` sensitivity -> (threshold, sensitivity)."""
    fpr, tpr, thresholds = roc_curve(y, score, drop_intermediate=False)
    idx = np.where(tpr >= target)[0][0]
    return thresholds[idx], tpr[idx]


def thresholds_by_group(val_data, group_col):
    """Validation thresholds at 90% / 95% sensitivity for each level of `group_col`."""
    rows = []
    for level, g in val_data.groupby(group_col):
        if g["actual"].nunique() < 2:
            continue
        thr90, sens90 = get_threshold(g["actual"], g["score"], 0.90)
        thr95, sens95 = get_threshold(g["actual"], g["score"], 0.95)
        rows.append({
            group_col: level,
            "N": len(g),
            "Positive_Cases": int(g["actual"].sum()),
            "Threshold_90": thr90,
            "Validation_Sensitivity_90": sens90 * 100,
            "Threshold_95": thr95,
            "Validation_Sensitivity_95": sens95 * 100,
        })
    return pd.DataFrame(rows).round(6)


def county_thresholds_validation(val_data, global_threshold_90, global_threshold_95,
                                 min_positives=MIN_POSITIVES):
    """
    County thresholds from VALIDATION data.
    Counties with more than `min_positives` positives get their own threshold;
    the rest use the frozen global validation thresholds.
    """
    rows = []
    for county in sorted(val_data["County"].dropna().unique()):
        county_df = val_data[val_data["County"] == county]
        positives = int(county_df["actual"].sum())

        if county_df["actual"].nunique() < 2:
            continue

        if positives > min_positives:
            threshold90, sensitivity90 = get_threshold(county_df["actual"], county_df["score"], 0.90)
            threshold95, sensitivity95 = get_threshold(county_df["actual"], county_df["score"], 0.95)
            sensitivity90, sensitivity95 = sensitivity90 * 100, sensitivity95 * 100
            source = "County-specific"
        else:
            threshold90, threshold95 = global_threshold_90, global_threshold_95
            pred90 = (county_df["score"] >= threshold90).astype(int)
            pred95 = (county_df["score"] >= threshold95).astype(int)
            actual_pos = county_df["actual"] == 1
            sensitivity90 = pred90[actual_pos].sum() / positives * 100 if positives > 0 else np.nan
            sensitivity95 = pred95[actual_pos].sum() / positives * 100 if positives > 0 else np.nan
            source = "Global"

        rows.append({
            "County": county,
            "N": len(county_df),
            "Positive_Cases": positives,
            "Threshold_Source": source,
            "Threshold_90": threshold90,
            "Validation_Sensitivity_90": sensitivity90,
            "Threshold_95": threshold95,
            "Validation_Sensitivity_95": sensitivity95,
        })
    return pd.DataFrame(rows).sort_values("County").reset_index(drop=True).round(6)

## 13. County-level evaluation with Wilson confidence intervals

In [ ]:
def thr_for_sens(y, score, target):
    """Threshold giving at least `target` sensitivity."""
    return get_threshold(y, score, target)[0]


def build_county_threshold_table(data, targets=TARGETS_PCT, min_pos=MIN_POSITIVES):
    """
    Global thresholds + county thresholds computed from `data`
    (needs `actual`, `score`, `County` columns).
    Returns (threshold_table, global_thr).
    """
    global_thr = {t: thr_for_sens(data["actual"], data["score"], v) for t, v in targets.items()}

    rows = []
    for county, g in data.groupby("County"):
        pos = int(g["actual"].sum())
        eligible = pos > min_pos and g["actual"].nunique() == 2
        row = {
            "County": county,
            "Positives": pos,
            "Threshold_Source": "County-specific" if eligible else "Global (grouped)",
        }
        for t, v in targets.items():
            row[f"Threshold_{t}"] = (
                thr_for_sens(g["actual"], g["score"], v) if eligible else global_thr[t]
            )
        rows.append(row)

    thr_table = pd.DataFrame(rows).sort_values("County").reset_index(drop=True)
    return thr_table, global_thr


def summarize_group(g, col, alpha=0.05):
    """Sensitivity (with Wilson CI), precision and share tested for one group."""
    tn, fp, fn, tp = confusion_matrix(g["actual"], g[col], labels=[0, 1]).ravel()
    pos = tp + fn
    if pos > 0:
        lo, hi = proportion_confint(tp, pos, alpha=alpha, method="wilson")
        sens = tp / pos * 100
    else:
        lo = hi = sens = np.nan
    return pd.Series({
        "N": len(g),
        "Positives": pos,
        "Sensitivity_%": sens,
        "Sens_CI_Lower_%": lo * 100,
        "Sens_CI_Upper_%": hi * 100,
        "Precision_%": tp / (tp + fp) * 100 if (tp + fp) else np.nan,
        "Share_Tested_%": (tp + fp) / len(g) * 100,
    })


def evaluate_county_performance(test_data, thr_table, global_thr, targets=TARGETS_PCT):
    """
    Apply the county / global thresholds in `thr_table` to `test_data`.
    Returns (test_data_with_predictions, perf) where perf[90] and perf[95] are county tables.
    """
    test_data = test_data.copy()
    thr_map = thr_table.set_index("County")
    perf = {}

    for t, target in targets.items():
        test_data[f"thr_{t}"] = (
            test_data["County"].map(thr_map[f"Threshold_{t}"]).fillna(global_thr[t])
        )
        test_data[f"pred_{t}"] = (test_data["score"] >= test_data[f"thr_{t}"]).astype(int)

        p = pd.DataFrame({
            county: summarize_group(g, f"pred_{t}")
            for county, g in test_data.groupby("County")
        }).T
        p.index.name = "County"
        p["Threshold_Source"] = thr_map["Threshold_Source"]
        p["CI_Includes_Target"] = (
            (p["Sens_CI_Lower_%"] <= target * 100) & (p["Sens_CI_Upper_%"] >= target * 100)
        )
        p["Lower_Bound_Meets_Target"] = p["Sens_CI_Lower_%"] >= target * 100
        perf[t] = p.round(2)

    return test_data, perf


def plot_county_ci(perf, test_data, targets=TARGETS_PCT,
                   save_path="county_sensitivity_CI_bands.png"):
    """Side-by-side sensitivity-by-county plots with 95% CI bands."""
    order = perf[95].sort_values("Sensitivity_%").index
    colors = {"County-specific": "#1f77b4", "Global (grouped)": "#ff7f0e"}

    fig, axes = plt.subplots(1, 2, figsize=(15, 11), sharey=True)
    for ax, (t, target) in zip(axes, targets.items()):
        d = perf[t].loc[order]
        y = np.arange(len(d))
        c = d["Threshold_Source"].map(colors)
        overall = summarize_group(test_data, f"pred_{t}")

        ax.axvspan(overall["Sens_CI_Lower_%"], overall["Sens_CI_Upper_%"],
                   color="grey", alpha=0.15, zorder=0)
        ax.axvline(overall["Sensitivity_%"], color="grey", lw=1, zorder=1)
        ax.barh(y, d["Sens_CI_Upper_%"] - d["Sens_CI_Lower_%"],
                left=d["Sens_CI_Lower_%"], height=0.6, color=c, alpha=0.35, zorder=2)
        ax.scatter(d["Sensitivity_%"], y, color=c, s=25, zorder=3)
        ax.axvline(target * 100, color="red", ls="--", lw=1.5, zorder=4)
        ax.set_yticks(y)
        ax.set_yticklabels(d.index, fontsize=8)
        ax.set_xlabel("Sensitivity (%) with 95% CI")
        ax.set_title(f"{t}% sensitivity target")
        ax.grid(axis="x", alpha=0.3)
        ax.set_xlim(max(0, d["Sens_CI_Lower_%"].min() - 2), 100.5)

    fig.legend(handles=[
        Patch(facecolor=colors["County-specific"], alpha=0.5, label="County-specific threshold (CI band)"),
        Patch(facecolor=colors["Global (grouped)"], alpha=0.5, label="Global threshold (CI band)"),
        Patch(facecolor="grey", alpha=0.25, label="Overall 95% CI"),
        Line2D([0], [0], color="red", ls="--", label="Target"),
    ], loc="lower center", ncol=4, frameon=False)
    fig.suptitle("Sensitivity by county with 95% CI bands (test set)", fontsize=14)
    plt.tight_layout(rect=[0, 0.04, 1, 0.97])
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    return fig

## 14. Save results to Excel

In [ ]:
def save_results_to_excel(sheets, filename="Final_Model_Thresholds_90_95.xlsx", download=True):
    """Write {sheet_name: DataFrame} to one workbook."""
    with pd.ExcelWriter(filename, engine="openpyxl") as writer:
        for sheet_name, df in sheets.items():
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    print(f"Saved: {filename}")
    if download:
        download_file(filename)
    return filename

## 15. Run the full pipeline
Set the flags below to control optional steps.

In [ ]:
def run_pipeline(
    data_paths=DATA_PATHS,
    save_and_download_models=True,
    load_saved_models=False,
    saved_xgb_path=os.path.join(MODEL_DIR, "xgb_model (1).pkl"),
    saved_best_path=os.path.join(MODEL_DIR, "best_model (1).pkl"),
    use_validation_thresholds_for_county_test=False,
    download_excel=True,
):
    """
    End-to-end run.
    - load_saved_models=True swaps the freshly trained models for the saved .pkl files
      (the original notebook did this unconditionally).
    - use_validation_thresholds_for_county_test=True applies county thresholds learned on
      VALIDATION to the test set (no test leakage). False reproduces the original code,
      which derived them from the test set.
    """
    # --- Data ---
    datasets = load_datasets(data_paths)
    datasets = drop_columns(datasets, COLS_TO_DROP)
    report_dtypes(datasets)

    X_train, y_train, X_val, y_val, X_test, y_test = split_features_target(datasets)
    cast_string_columns_to_object(datasets)
    print_unique_counts(X_train)

    # --- Encoding ---
    encoder, X_train_enc, X_val_enc, X_test_enc, _ = encode_features(X_train, X_val, X_test)
    describe_sparse_matrix(X_train_enc)
    _, y_train_enc, y_val_enc, y_test_enc = encode_target(y_train, y_val, y_test)
    compute_scale_pos_weight(y_train_enc)

    # --- Training ---
    best_model, last_model, best_params, best_score, results_df = train_xgboost_grid(
        X_train_enc, y_train_enc, X_val_enc, y_val_enc
    )
    if save_and_download_models:
        save_models({"xgb_modelupdated.pkl": last_model,
                     "best_modelupdated.pkl": best_model})
    if load_saved_models:
        last_model, best_model = load_models(saved_xgb_path, saved_best_path)

    # --- Validation thresholds ---
    evaluate_default_threshold(best_model, X_val_enc, y_val_enc)
    y_val_prob = best_model.predict_proba(X_val_enc)[:, 1]
    operating_points_val = find_operating_points(y_val_enc, y_val_prob)
    threshold_90, threshold_95 = freeze_thresholds(operating_points_val)
    global_thresholds_df = build_global_thresholds_df(operating_points_val)

    # --- Test evaluation ---
    y_test_prob = best_model.predict_proba(X_test_enc)[:, 1]
    report_test_performance(y_test_enc, y_test_prob, threshold_90, "90%")
    report_test_performance(y_test_enc, y_test_prob, threshold_95, "95%")

    y_pred_90, y_pred_95 = plot_pr_curve(y_test_enc, y_test_prob, threshold_90, threshold_95)
    plot_confusion_matrix(y_test_enc, y_pred_90, "Confusion Matrix - 90% Sensitivity Target", "Blues")
    plot_confusion_matrix(y_test_enc, y_pred_95, "Confusion Matrix - 95% Sensitivity Target", "Greens")
    test_positivity_and_yield(y_test_enc, y_test_prob,
                              [(0.90, threshold_90), (0.95, threshold_95)])

    # --- Feature importance ---
    importance_df, feature_summary = summarize_feature_importance(encoder, best_model)
    print(feature_summary.head(20))

    # --- Group-specific thresholds (validation) ---
    val_data = build_scored_frame(datasets["sparse_val"], y_val_enc, best_model, X_val_enc)
    check_scored_frame(val_data, X_val_enc, y_val_enc)
    sex_thresholds_df = thresholds_by_group(val_data, "Sex")
    population_thresholds_df = thresholds_by_group(val_data, "PopulationType")
    county_thresholds_df = county_thresholds_validation(val_data, threshold_90, threshold_95)

    # --- County evaluation on test ---
    test_data = build_scored_frame(datasets["sparse_test"], y_test_enc, best_model, X_test_enc)
    threshold_source = val_data if use_validation_thresholds_for_county_test else test_data
    thr_table, global_thr = build_county_threshold_table(threshold_source)
    test_data, perf = evaluate_county_performance(test_data, thr_table, global_thr)
    plot_county_ci(perf, test_data)
    print(perf[90])
    print(perf[95])

    # --- Export ---
    save_results_to_excel(
        {
            "Global_Thresholds": global_thresholds_df,
            "Sex_Thresholds": sex_thresholds_df,
            "PopulationType_Thresholds": population_thresholds_df,
            "County_Thresholds": county_thresholds_df,
        },
        download=download_excel,
    )

    return {
        "best_model": best_model, "best_params": best_params, "results_df": results_df,
        "encoder": encoder, "thresholds": {"90": threshold_90, "95": threshold_95},
        "feature_summary": feature_summary,
        "sex_thresholds": sex_thresholds_df,
        "population_thresholds": population_thresholds_df,
        "county_thresholds": county_thresholds_df,
        "county_perf": perf,
    }

In [ ]:
results = run_pipeline()